# NER for Sentences

### Authors - Shivendra Raghav (1001123853), Amirhosein Morteza (1002187551)
### CSE 5388-002

#### This file trains the NER model for the sentences.

#### References: https://spacy.io/usage/training, https://spacy.io/api/scorer

### Import the required libraries

In [99]:
import spacy
import random
import pandas as pd
from spacy.scorer import Scorer
from spacy.training import Example
from spacy.util import minibatch, compounding
from sklearn.model_selection import train_test_split

### Processing the NER metadata

In [100]:
# Reading the NER Sentence Dataset
df = pd.read_csv("../synthetic_dataset/sentence_classified_data_ner.csv")

# Filter the sensitive sentences
df_sensitive = df[df["sensitive"] == 1]

# Load the spaCy model and create a blank EntityRecognizer
nlp = spacy.blank("en")
ner = nlp.add_pipe("ner", last=True)

### Conver into the required format and split the dataset

In [101]:
# Convert the dataset into a list of tuples for NER training
ner_metadata = [(row["sentence"], {"entities": eval(row["ner_metadata"])}) for _, row in df_sensitive.iterrows()]

# Split the data into training and testing sets
train_data, test_data = train_test_split(ner_metadata, test_size=0.2, random_state=42)

### Adding all the labels to the NER model

In [102]:
# Adding our labels to the EntityRecognizer
for _, annotations in train_data:
    for entity in annotations.get("entities"):
        ner.add_label(entity[2])

ner.labels

('API_KEY', 'DATABASE_CONN', 'PASSWORD', 'SSN', 'USERNAME')

### SpaCy NER model Training

In [103]:
optimizer = nlp.initialize()
for itn in range(10):
    random.shuffle(train_data)
    losses = {}
    for text, annotations in train_data:
        doc = nlp.make_doc(text)
        example = Example.from_dict(doc, annotations)
        nlp.update([example], sgd=optimizer, losses=losses)
    print("Iteration:", itn + 1, "Losses:", losses)

Iteration: 1 Losses: {'ner': 1697.6256584594767}
Iteration: 2 Losses: {'ner': 544.8660801515699}
Iteration: 3 Losses: {'ner': 324.681593788379}
Iteration: 4 Losses: {'ner': 190.29062063628564}
Iteration: 5 Losses: {'ner': 229.19854767146697}
Iteration: 6 Losses: {'ner': 171.80218510692123}
Iteration: 7 Losses: {'ner': 150.47027403702657}
Iteration: 8 Losses: {'ner': 147.7125902708399}
Iteration: 9 Losses: {'ner': 113.76042343827267}
Iteration: 10 Losses: {'ner': 96.89644356794405}


### Evalutation of the NER model 

In [104]:
# Evaluate the trained model using Scorer
scorer = Scorer(nlp)
examples = []
for text, annotations in test_data:
    doc = nlp(text)
    example = Example.from_dict(doc, annotations)
    scorer.score([example])
    examples.append(example)

scores = scorer.score(examples)

In [105]:
# Printing the evaluation metrics
df_scores = pd.DataFrame(columns=["Metric", "Value"])
df_scores = df_scores._append({"Metric": "Precision", "Value":scores['ents_p']}, ignore_index=True)
df_scores = df_scores._append({"Metric": "Recall", "Value":scores['ents_r']}, ignore_index=True)
df_scores = df_scores._append({"Metric": "F1 score:", "Value":scores['ents_f']}, ignore_index=True)

df_scores.head()

/var/folders/rz/4djzw0r155g8_kdwpx3gm7y40000gn/T/ipykernel_71007/3692532330.py:3: FutureWarning: The behavior of DataFrame concatenation with empty or all-NA entries is deprecated. In a future version, this will no longer exclude empty or all-NA columns when determining the result dtypes. To retain the old behavior, exclude the relevant entries before the concat operation.
  df_scores = df_scores._append({"Metric": "Precision", "Value":scores['ents_p']}, ignore_index=True)


,Metric,Value
0,Precision,0.881443
1,Recall,0.686747
2,F1 score:,0.772009
